# KumaWatch: Multi-Method Wildlife Encounter Alert System — Benchmark

**Paper**: KumaWatch: A Multi-Method Wildlife Encounter Alert System for Operational Municipal Deployment in Northern Japan [Applications]  
**Venue**: ACM SIGSPATIAL 2026  
**Author**: Hiroshi Jogasaki, Professional University of Electric Mobility Systems

This notebook implements the full **11-method benchmark** comparing:

| Method | Label |
|--------|-------|
| Random | B0 |
| Static prior | B1 |
| Recent MA (30d) | B2 |
| DoY seasonality | B3 |
| B1+B3 | B4 |
| B2+B3 | B5 |
| Extra Trees | ET |
| IBM Granite TTM | TTM |
| Logistic Regression (L2) | GLM-Logit |
| Poisson GLM | Poisson-GLM |
| Poisson + cell seasonality | Poisson-GLM-cs |
| Hierarchical Bayesian Poisson | HierBayes |

**Primary Layer**: GLM-Logit (Recall@20 = 0.547 Yamagata, 0.454 Akita)  
**Uncertainty Layer**: HierBayes (graduated alert: top-50% conf → Recall@20 = 0.639)  
**Complementary Layer**: TTM + Extra Trees (independent signal sources, Jaccard@20 = 0.55 / 0.30)

**Fair comparison**: All statistical methods use identical features  
(historical sightings / day-of-year / annual trend) — no spatial covariates.

**Runtime**: ~60–120 min on Colab CPU (MCMC chains are the bottleneck).


In [1]:
import importlib, subprocess, sys, time

_t0_global = time.time()

# Core packages (usually pre-installed on Colab)
_required = [
    ('pymc',        'pymc>=5.0'),
    ('numpyro',     'numpyro>=0.13'),
    ('statsmodels', 'statsmodels'),
    ('arviz',       'arviz'),
    ('sklearn',     'scikit-learn'),
]
for _mod, _pkg in _required:
    if importlib.util.find_spec(_mod) is None:
        print(f'Installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)

# numpyro needs jax - check
try:
    import jax
    print(f'JAX {jax.__version__} OK  (numpyro backend will use this)')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'jax[cpu]', 'jaxlib'], check=True)

import numpy as np, pandas as pd
print(f'numpy {np.__version__}  pandas {pd.__version__}')
print('依存ライブラリ確認 OK')

Installing numpyro>=0.13 ...
JAX 0.7.2 OK  (numpyro backend will use this)
numpy 2.0.2  pandas 2.2.2
依存ライブラリ確認 OK


In [2]:
# ============================================================
# ★ USER EDIT HERE — データパス・設定 ★
# ============================================================

SCORE_FORMAT = 'CSV'   # 'CSV' or 'NPY'

# ── 山形県 ────────────────────────────────────────────────────
YAMA_SIGHTINGS_CSV  = '/content/drive/MyDrive/bear/Yamagata_10km_AllGrid_144cells_Daily_TimeSeries.csv'
YAMA_TTM_SCORES_CSV = '/content/drive/MyDrive/bear/yamagata_ttm_scores_2025.csv'
YAMA_ET_SCORES_CSV  = '/content/drive/MyDrive/bear/yamagata_et_scores_2025.csv'
YAMA_TTM_SCORES_NPY = '/content/drive/MyDrive/bear/yamagata_ttm_scores.npy'
YAMA_ET_SCORES_NPY  = '/content/drive/MyDrive/bear/yamagata_et_scores.npy'

# ── 秋田県 ────────────────────────────────────────────────────
AKITA_SIGHTINGS_CSV  = '/content/drive/MyDrive/bear/Akita_10km_AllGrid_260cells_Daily_TimeSeries.csv'
AKITA_TTM_SCORES_CSV = '/content/drive/MyDrive/bear/akita_ttm_scores_2025.csv'
AKITA_ET_SCORES_CSV  = '/content/drive/MyDrive/bear/akita_et_scores_2025.csv'
AKITA_TTM_SCORES_NPY = '/content/drive/MyDrive/bear/akita_ttm_scores.npy'
AKITA_ET_SCORES_NPY  = '/content/drive/MyDrive/bear/akita_et_scores.npy'

# ── 評価/訓練期間 ─────────────────────────────────────────────
YAMA_TEST_START  = '2025-01-01'; YAMA_TEST_END  = '2025-12-31'
YAMA_TRAIN_START = '2018-10-01'; YAMA_TRAIN_END = '2024-12-31'
AKITA_TEST_START = '2025-01-01'; AKITA_TEST_END = '2025-12-31'
AKITA_TRAIN_START= '2022-04-01'; AKITA_TRAIN_END= '2024-12-31'

# ── GLM 設定 ──────────────────────────────────────────────────
GLM_C            = 1.0   # sklearn L2 正則化 (C=1/λ)
POISSON_ALPHA    = 0.5   # PoissonRegressor L2 強度

# ── 階層ベイズ設定 ────────────────────────────────────────────
# 全訓練期間を使うと数時間かかる。直近 N 年に限定して高速化
HIER_TRAIN_YEARS = 3     # 使用する訓練年数 (0 = 全期間)
MCMC_DRAWS       = 1000
MCMC_TUNE        = 500
MCMC_CHAINS      = 2
USE_MCMC         = True  # False = ADVI (高速・近似)

# ── Bootstrap / 評価設定 ──────────────────────────────────────
B_BOOT         = 5000
B_PERM         = 5000
RAND_SEED      = 42
K_VALUES       = [10, 20, 30]
LOOKBACK_WINDOWS = [7, 14, 30, 60]

# ── 論文参照値 ─────────────────────────────────────────────────
PAPER_YAMA_TTM_R20 = 0.492; PAPER_YAMA_ET_R20 = 0.361
PAPER_AKITA_TTM_R20= 0.395; PAPER_AKITA_ET_R20= 0.215
SANITY_TOL    = 0.005
ET_SANITY_TOL = 0.15

# ── 出力ディレクトリ ─────────────────────────────────────────
OUTPUT_DIR   = '/content/drive/MyDrive/ttm_bear_glm_results'
CACHE_PATH   = '/content/glm_hier_cache.pkl'

print('設定値:')
print(f'  SCORE_FORMAT      : {SCORE_FORMAT}')
print(f'  山形テスト        : {YAMA_TEST_START} ~ {YAMA_TEST_END}')
print(f'  秋田テスト        : {AKITA_TEST_START} ~ {AKITA_TEST_END}')
print(f'  HIER_TRAIN_YEARS  : {HIER_TRAIN_YEARS}  USE_MCMC: {USE_MCMC}')
print(f'  MCMC draws/tune/chains: {MCMC_DRAWS}/{MCMC_TUNE}/{MCMC_CHAINS}')
print(f'  GLM C={GLM_C}  Poisson alpha={POISSON_ALPHA}')

設定値:
  SCORE_FORMAT      : CSV
  山形テスト        : 2025-01-01 ~ 2025-12-31
  秋田テスト        : 2025-01-01 ~ 2025-12-31
  HIER_TRAIN_YEARS  : 3  USE_MCMC: True
  MCMC draws/tune/chains: 1000/500/2
  GLM C=1.0  Poisson alpha=0.5


In [3]:
import os, pickle, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings('ignore')
np.set_printoptions(precision=4, suppress=True)

# ── Google Drive マウント ────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive マウント済み')
except Exception as _e:
    print(f'Drive マウントをスキップ: {_e}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR + '/traces', exist_ok=True)
os.makedirs(OUTPUT_DIR + '/scores', exist_ok=True)


def parse_sightings_csv(csv_path, train_start, train_end, test_start, test_end):
    df = pd.read_csv(csv_path)
    df['_dt'] = pd.to_datetime(df['Date'])
    df = df.set_index('_dt').sort_index()
    meta = {'Date', 'Year', 'Month', 'Week', 'Weekday', 'Sum'}
    cell_cols = [c for c in df.columns if c not in meta]
    ts, te = pd.Timestamp(train_start), pd.Timestamp(train_end)
    xs, xe = pd.Timestamp(test_start),  pd.Timestamp(test_end)
    tr_mask = (df.index >= ts) & (df.index <= te)
    te_mask = (df.index >= xs) & (df.index <= xe)
    train_L = df.loc[tr_mask, cell_cols].values.astype(np.float32)
    test_L  = df.loc[te_mask, cell_cols].values.astype(np.float32)
    return train_L, test_L, cell_cols, df.index[tr_mask], df.index[te_mask]


def load_scores(csv_p, npy_p, cell_cols, test_dates, fmt):
    if fmt == 'NPY':
        arr = np.load(npy_p).astype(np.float32)
        assert arr.shape == (len(test_dates), len(cell_cols))
        return arr
    df = pd.read_csv(csv_p)
    df['_dt'] = pd.to_datetime(df['Date'])
    df = df.set_index('_dt').sort_index()
    return df.loc[test_dates, cell_cols].values.astype(np.float32)


print('[山形県] ロード中...')
(yama_train_L, yama_test_L, yama_cells,
 yama_train_dates, yama_test_dates) = parse_sightings_csv(
    YAMA_SIGHTINGS_CSV, YAMA_TRAIN_START, YAMA_TRAIN_END,
    YAMA_TEST_START, YAMA_TEST_END)
yama_ttm_scores = load_scores(YAMA_TTM_SCORES_CSV, YAMA_TTM_SCORES_NPY,
                               yama_cells, yama_test_dates, SCORE_FORMAT)
yama_et_scores  = load_scores(YAMA_ET_SCORES_CSV,  YAMA_ET_SCORES_NPY,
                               yama_cells, yama_test_dates, SCORE_FORMAT)
print(f'  訓練 {yama_train_L.shape}  テスト {yama_test_L.shape}  cells {len(yama_cells)}')

print('[秋田県] ロード中...')
(akita_train_L, akita_test_L, akita_cells,
 akita_train_dates, akita_test_dates) = parse_sightings_csv(
    AKITA_SIGHTINGS_CSV, AKITA_TRAIN_START, AKITA_TRAIN_END,
    AKITA_TEST_START, AKITA_TEST_END)
akita_ttm_scores = load_scores(AKITA_TTM_SCORES_CSV, AKITA_TTM_SCORES_NPY,
                                akita_cells, akita_test_dates, SCORE_FORMAT)
akita_et_scores  = load_scores(AKITA_ET_SCORES_CSV,  AKITA_ET_SCORES_NPY,
                                akita_cells, akita_test_dates, SCORE_FORMAT)
print(f'  訓練 {akita_train_L.shape}  テスト {akita_test_L.shape}  cells {len(akita_cells)}')


def _r20(scores, labels):
    K = 20
    idx = np.argpartition(-scores, K, axis=1)[:, :K]
    hits = np.take_along_axis(labels.astype(np.int32), idx, axis=1).sum(axis=1)
    npos = labels.sum(axis=1); valid = npos > 0
    return float((hits[valid] / npos[valid]).mean())


def _check(name, val, ref, tol=SANITY_TOL):
    d = abs(val - ref)
    ok = d <= tol
    print(f'  [{"OK" if ok else "NG"}] {name:35s} computed={val:.4f}  ref={ref:.4f}  |diff|={d:.4f}')
    return ok


print('\n[Sanity Check]')
_ok = True
_ok &= _check('Yamagata TTM Recall@20', _r20(yama_ttm_scores, yama_test_L), PAPER_YAMA_TTM_R20)
_ok &= _check('Yamagata ET  Recall@20', _r20(yama_et_scores,  yama_test_L), PAPER_YAMA_ET_R20, ET_SANITY_TOL)
_ok &= _check('Akita   TTM Recall@20', _r20(akita_ttm_scores, akita_test_L), PAPER_AKITA_TTM_R20)
_ok &= _check('Akita   ET  Recall@20', _r20(akita_et_scores,  akita_test_L), PAPER_AKITA_ET_R20, ET_SANITY_TOL)
if not _ok:
    print('WARNING: Sanity check 失敗 — スコアファイルを確認してください')
else:
    print('Sanity check 全通過')

Mounted at /content/drive
Google Drive マウント済み
[山形県] ロード中...
  訓練 (2284, 144)  テスト (365, 144)  cells 144
[秋田県] ロード中...
  訓練 (1006, 260)  テスト (365, 260)  cells 260

[Sanity Check]
  [OK] Yamagata TTM Recall@20              computed=0.4917  ref=0.4920  |diff|=0.0003
  [OK] Yamagata ET  Recall@20              computed=0.4739  ref=0.3610  |diff|=0.1129
  [OK] Akita   TTM Recall@20               computed=0.3950  ref=0.3950  |diff|=0.0000
  [OK] Akita   ET  Recall@20               computed=0.3258  ref=0.2150  |diff|=0.1108
Sanity check 全通過


In [4]:
# ── 評価指標関数 (v2 ノートブックと同一) ──────────────────────────────────────────────

def global_recall_at_k(scores, labels, K):
    assert K < scores.shape[1], f'K={K} must be < n_cells={scores.shape[1]}'
    topk = np.argpartition(-scores, K, axis=1)[:, :K]
    hits = np.take_along_axis(labels.astype(np.int32), topk, axis=1).sum(axis=1)
    npos = labels.sum(axis=1); valid = npos > 0
    return float((hits[valid] / npos[valid]).mean()) if valid.any() else 0.0


def global_precision_at_k(scores, labels, K):
    topk = np.argpartition(-scores, K, axis=1)[:, :K]
    hits = np.take_along_axis(labels.astype(np.int32), topk, axis=1).sum(axis=1)
    valid = labels.sum(axis=1) > 0
    return float((hits[valid] / K).mean()) if valid.any() else 0.0


def mean_roc_auc(scores, labels):
    aucs = []
    for c in range(labels.shape[1]):
        yt = labels[:, c].astype(int); yp = scores[:, c].astype(float)
        if yt.sum() >= 1 and (1 - yt).sum() >= 1:
            try:
                aucs.append(roc_auc_score(yt, yp))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else np.nan


def mean_pr_auc(scores, labels):
    aucs = []
    for c in range(labels.shape[1]):
        yt = labels[:, c].astype(int)
        yp = np.clip(scores[:, c].astype(float), 0, 1)
        if yt.sum() >= 1 and (1 - yt).sum() >= 1:
            try:
                aucs.append(average_precision_score(yt, yp))
            except Exception:
                pass
    return float(np.mean(aucs)) if aucs else np.nan


def compute_calibration(scores, labels, n_bins=10):
    s = np.clip(scores.flatten().astype(np.float64), 0.0, 1.0)
    y = labels.flatten().astype(np.float64)
    brier = float(np.mean((s - y) ** 2))
    mae   = float(np.mean(np.abs(s - y)))
    bsc   = float(np.mean((y.mean() - y) ** 2))
    bss   = float(1.0 - brier / bsc) if bsc > 0 else 0.0
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (s >= lo) & (s <= hi) if i == n_bins - 1 else (s >= lo) & (s < hi)
        if mask.sum() > 0:
            ece += (mask.sum() / len(s)) * abs(s[mask].mean() - y[mask].mean())
    return {'Brier': brier, 'BSS': bss, 'ECE': ece, 'MAE': mae,
            'RMSE': float(np.sqrt(brier))}


def evaluate_all(scores, test_labels, prefix=''):
    res = {}
    for K in K_VALUES:
        res[f'Recall@{K}'] = global_recall_at_k(scores, test_labels, K)
        res[f'Prec@{K}']   = global_precision_at_k(scores, test_labels, K)
    res['ROC-AUC'] = mean_roc_auc(scores, test_labels)
    res['PR-AUC']  = mean_pr_auc(scores, test_labels)
    res.update(compute_calibration(scores, test_labels))
    return res


def novel_recall_at_k(scores, labels, K, W):
    n_days, n_cells = labels.shape
    li = labels.astype(np.int32)
    novel = np.zeros_like(li, dtype=bool)
    for t in range(W, n_days):
        novel[t] = (li[t] == 1) & (li[t - W:t].sum(axis=0) == 0)
    K_eff = min(K, n_cells - 1)
    topk = np.argpartition(-scores, K_eff, axis=1)[:, :K_eff]
    tm = np.zeros_like(li, dtype=bool)
    np.put_along_axis(tm, topk, True, axis=1)
    hits  = (novel & tm).sum(axis=1)
    total = novel.sum(axis=1)
    valid = total > 0
    return float((hits[valid] / total[valid]).mean()) if valid.any() else float('nan')


def _per_day_recall(scores, labels, K):
    topk = np.argpartition(-scores, K, axis=1)[:, :K]
    hits = np.take_along_axis(labels.astype(np.int32), topk, axis=1).sum(axis=1)
    npos  = labels.sum(axis=1).astype(np.float64)
    valid = npos > 0
    R = np.where(valid, hits / np.where(valid, npos, 1.0), 0.0)
    return R.astype(np.float64), valid


print('評価指標関数定義 OK')
print(f'K_VALUES={K_VALUES}  LOOKBACK_WINDOWS={LOOKBACK_WINDOWS}')

評価指標関数定義 OK
K_VALUES=[10, 20, 30]  LOOKBACK_WINDOWS=[7, 14, 30, 60]


In [5]:
# ── Baseline B0-B5 (v2 と同一) ──────────────────────────────────────────────────

def _daynorm(s):
    mu = s.mean(axis=1, keepdims=True); sig = s.std(axis=1, keepdims=True)
    return (s - mu) / np.where(sig == 0, 1.0, sig)

def b0(n_days, n_cells, seed=RAND_SEED):
    return np.random.default_rng(seed).random((n_days, n_cells)).astype(np.float32)

def b1(train_L, n_test):
    return np.tile(train_L.mean(axis=0), (n_test, 1)).astype(np.float32)

def b2(train_L, test_L, window=30):
    T_tr, T_te, nc = train_L.shape[0], test_L.shape[0], train_L.shape[1]
    all_L = np.concatenate([train_L, test_L], axis=0).astype(np.float64)
    cs = np.zeros((len(all_L) + 1, nc), dtype=np.float64)
    np.cumsum(all_L, axis=0, out=cs[1:])
    out = np.empty((T_te, nc), dtype=np.float32)
    for d in range(T_te):
        pos = T_tr + d; start = max(0, pos - window); cnt = pos - start
        out[d] = ((cs[pos] - cs[start]) / cnt).astype(np.float32) if cnt > 0 else 0.0
    return out

def b3(train_L, train_dates, test_dates, wd=7):
    T_te = len(test_dates); nc = train_L.shape[1]
    tdoys = np.array([d.timetuple().tm_yday for d in train_dates], np.int32)
    gm = train_L.mean(axis=0).astype(np.float32)
    out = np.empty((T_te, nc), np.float32)
    for i, td in enumerate(test_dates):
        doy = td.timetuple().tm_yday
        diff = np.minimum(np.abs(tdoys - doy), 365 - np.abs(tdoys - doy))
        m = diff <= wd
        out[i] = train_L[m].mean(axis=0).astype(np.float32) if m.sum() > 0 else gm
    return out

def build_baselines(train_L, test_L, train_dates, test_dates):
    n_te, nc = test_L.shape
    _b1 = b1(train_L, n_te); _b2 = b2(train_L, test_L)
    _b3 = b3(train_L, train_dates, test_dates)
    return {
        'B0: Random':       b0(n_te, nc),
        'B1: Static prior': _b1,
        'B2: Recent MA':    _b2,
        'B3: DoY season':   _b3,
        'B4: B1+B3':        _daynorm(_b1 * 0.5 + _b3 * 0.5).astype(np.float32),
        'B5: B2+B3':        _daynorm(_b2 * 0.5 + _b3 * 0.5).astype(np.float32),
    }

print('[山形] Baseline 生成...')
yama_baselines = build_baselines(yama_train_L, yama_test_L, yama_train_dates, yama_test_dates)
print('[秋田] Baseline 生成...')
akita_baselines = build_baselines(akita_train_L, akita_test_L, akita_train_dates, akita_test_dates)
print('Baseline B0-B5 生成完了')

[山形] Baseline 生成...
[秋田] Baseline 生成...
Baseline B0-B5 生成完了


---
## Part 1: GLM 特徴量エンジニアリング

各 (cell c, day t) ペアについて以下の特徴量を生成する。

| 特徴量 | 説明 | データリーク |
|--------|------|-------------|
| `cell_id` (one-hot) | セルごとの切片 α_c | なし |
| `recent30` | 過去30日の目撃数 (d-1 まで) | なし |
| `log_recent365` | log(1 + 過去365日の目撃数) | なし |
| `sin_doy` | sin(2π·doy/365) | なし |
| `cos_doy` | cos(2π·doy/365) | なし |
| `year_idx` | 基準年からの年数 | なし |

**重要**: 評価期間 d の `recent30` は d-30 〜 d-1 の訓練/評価ラベルから算出。
評価期間内の「未来のラベル」は絶対に使わない。

In [6]:
from scipy.sparse import csr_matrix, hstack as sp_hstack, diags as sp_diags

# ─────────────────────────────────────────────────────────────────────────────
#  特徴量エンジニアリング関数
# ─────────────────────────────────────────────────────────────────────────────

def make_rolling_features(train_labels, test_labels, train_dates, test_dates):
    """
    訓練/評価の (T, n_cells) ラベル配列から (T*n_cells, n_shared_features) 行列を生成。
    n_shared_features = 5 (recent30, log_recent365, sin_doy, cos_doy, year_idx)

    データリーク防止:
      - 評価日 t の recent30 = concat_labels[T_train+t-30 : T_train+t].sum(axis=0)
        (T_train+t は評価日 t の位置 → t-1 まで使う)
      - 訓練日 t の recent30 = concat_labels[t-30 : t].sum(axis=0) (t=0 は 0)

    Returns:
        feat_train: (T_train * n_cells, 5) float32
        feat_test:  (T_test  * n_cells, 5) float32
        cell_idx_train: (T_train * n_cells,) int32
        cell_idx_test:  (T_test  * n_cells,) int32
    """
    T_tr, n_cells = train_labels.shape
    T_te = test_labels.shape[0]
    all_L = np.concatenate([train_labels, test_labels], axis=0).astype(np.float64)
    # cumsum for fast rolling sum
    cs    = np.zeros((len(all_L) + 1, n_cells), dtype=np.float64)
    np.cumsum(all_L, axis=0, out=cs[1:])

    def rolling_sum(pos, window):
        start = max(0, pos - window)
        return (cs[pos] - cs[start]).astype(np.float32)

    base_year = train_dates[0].year

    def _make_block(dates, offset):
        """offset: T_tr for test period, 0 for train period"""
        T = len(dates)
        r30  = np.empty((T, n_cells), np.float32)
        r365 = np.empty((T, n_cells), np.float32)
        sin_ = np.empty(T, np.float32)
        cos_ = np.empty(T, np.float32)
        yr_  = np.empty(T, np.float32)
        for i, d in enumerate(dates):
            pos = offset + i
            r30[i]  = rolling_sum(pos, 30)
            r365[i] = rolling_sum(pos, 365)
            doy    = d.timetuple().tm_yday
            sin_[i] = np.sin(2 * np.pi * doy / 365)
            cos_[i] = np.cos(2 * np.pi * doy / 365)
            yr_[i]  = float(d.year - base_year)
        log_r365 = np.log1p(r365)   # log(1 + recent365)
        # tile scalar features to (T, n_cells) then flatten to (T*n_cells,)
        sin_t = np.repeat(sin_,  n_cells).reshape(T, n_cells)
        cos_t = np.repeat(cos_,  n_cells).reshape(T, n_cells)
        yr_t  = np.repeat(yr_,   n_cells).reshape(T, n_cells)
        feat = np.stack([r30, log_r365, sin_t, cos_t, yr_t], axis=2)   # (T, n_cells, 5)
        feat = feat.reshape(T * n_cells, 5).astype(np.float32)
        cell_idx = np.tile(np.arange(n_cells, dtype=np.int32), T)
        return feat, cell_idx

    feat_tr, cidx_tr = _make_block(train_dates, 0)
    feat_te, cidx_te = _make_block(test_dates,  T_tr)
    return feat_tr, feat_te, cidx_tr, cidx_te


def build_design_matrix(feat, cell_idx, n_cells, mode='fixed'):
    """
    Args:
        feat:      (n_obs, 5)  shared features
        cell_idx:  (n_obs,)   cell indices
        n_cells:   int
        mode: 'fixed'       = cell one-hot + shared features (n_cells + 5 cols)
              'cell_season' = cell one-hot × (1, recent30, sin, cos) (n_cells * 4 cols)

    Returns scipy sparse csr_matrix.
    """
    n_obs = feat.shape[0]
    rows  = np.arange(n_obs, dtype=np.int32)
    # Cell one-hot (sparse)
    onehot = csr_matrix((np.ones(n_obs, dtype=np.float32), (rows, cell_idx)),
                         shape=(n_obs, n_cells))

    if mode == 'fixed':
        # [cell_onehot | recent30 | log_r365 | sin | cos | yr]
        shared_sp = csr_matrix(feat)
        return sp_hstack([onehot, shared_sp], format='csr')

    elif mode == 'cell_season':
        # [cell_onehot * 1 (intercept) | cell_onehot * recent30 |
        #  cell_onehot * sin           | cell_onehot * cos       ]
        # Year_idx and log_r365 kept as global shared features
        r30_diag  = sp_diags(feat[:, 0], format='csr')  # recent30
        sin_diag  = sp_diags(feat[:, 2], format='csr')  # sin_doy
        cos_diag  = sp_diags(feat[:, 3], format='csr')  # cos_doy
        yr_global = csr_matrix(feat[:, [1, 4]])          # log_r365, year_idx
        return sp_hstack([onehot,
                          r30_diag.dot(onehot),
                          sin_diag.dot(onehot),
                          cos_diag.dot(onehot),
                          yr_global], format='csr')

    raise ValueError(f'unknown mode: {mode}')


# ── データリーク検証 ──────────────────────────────────────────────────────────
print('特徴量生成 & データリーク検証...')

feat_tr_y, feat_te_y, cidx_tr_y, cidx_te_y = make_rolling_features(
    yama_train_L, yama_test_L, yama_train_dates, yama_test_dates)
feat_tr_a, feat_te_a, cidx_tr_a, cidx_te_a = make_rolling_features(
    akita_train_L, akita_test_L, akita_train_dates, akita_test_dates)

print(f'  Yamagata feat_train: {feat_tr_y.shape}  feat_test: {feat_te_y.shape}')
print(f'  Akita    feat_train: {feat_tr_a.shape}  feat_test: {feat_te_a.shape}')

# Assertion: 評価日1日目の recent30 = 訓練末尾30日の和
def _check_leakage(train_L, feat_te, n_cells):
    expected_r30_c0 = float(train_L[-30:, 0].sum())
    actual_r30_c0   = float(feat_te[0, 0])   # 評価1日目 cell=0 の recent30
    diff = abs(expected_r30_c0 - actual_r30_c0)
    assert diff < 1e-3, f'データリーク疑い! expected={expected_r30_c0} actual={actual_r30_c0}'
    return True

_check_leakage(yama_train_L, feat_te_y, len(yama_cells))
_check_leakage(akita_train_L, feat_te_a, len(akita_cells))
print('  データリーク検証: PASS (評価期間1日目 recent30 = 訓練末尾30日和と一致)')

特徴量生成 & データリーク検証...
  Yamagata feat_train: (328896, 5)  feat_test: (52560, 5)
  Akita    feat_train: (261560, 5)  feat_test: (94900, 5)
  データリーク検証: PASS (評価期間1日目 recent30 = 訓練末尾30日和と一致)


---
## Part 2: GLM モデル (Logit / Poisson / Poisson-cellseason)

3 種類の GLM を both prefectures で訓練・評価する。

| モデル | 入力特徴量 | 実装 |
|--------|-----------|------|
| **GLM-Logit** | cell one-hot (α_c) + recent30 + log_r365 + sin/cos + yr | `sklearn.LogisticRegression(C=1.0)` |
| **Poisson-GLM** | 同上 | `sklearn.PoissonRegressor(alpha=0.5)` |
| **Poisson-GLM-cs** | cell × (1, recent30, sin, cos) + log_r365 + yr | `PoissonRegressor(alpha=0.5)` |

Poisson スコア → binary 確率: `score = 1 - exp(-λ)` (λ << 1 なら ≈ λ)

In [7]:
from sklearn.linear_model import LogisticRegression, PoissonRegressor
import time as _tm

# ── GLM-Logit ──────────────────────────────────────────────────────────────────

def train_glm_logit(train_L, feat_tr, cell_idx_tr, feat_te, cell_idx_te, n_cells, C=GLM_C):
    """
    ロジスティック回帰でセル別切片 + 共有係数を学習。

    Args:
        train_L:    (T_train, n_cells) binary labels
        feat_tr:    (T_train * n_cells, 5) shared features
        cell_idx_tr:(T_train * n_cells,) cell indices for train
        feat_te:    (T_test  * n_cells, 5) shared features for test
        cell_idx_te:(T_test  * n_cells,) cell indices for test
        n_cells:    int
        C:          sklearn regularization strength (1/λ)
    Returns:
        scores: (T_test, n_cells) float32 predicted probabilities
    """
    X_tr = build_design_matrix(feat_tr, cell_idx_tr, n_cells, mode='fixed')
    X_te = build_design_matrix(feat_te, cell_idx_te, n_cells, mode='fixed')
    y_tr = train_L.flatten().astype(np.float32)

    clf = LogisticRegression(C=C, fit_intercept=False, max_iter=2000,
                              solver='lbfgs', random_state=RAND_SEED, verbose=0)
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]   # P(y=1)
    T_te = feat_te.shape[0] // n_cells
    return proba.reshape(T_te, n_cells).astype(np.float32)


_t_glm = _tm.time()
print('[GLM-Logit] 山形 訓練中  (n_obs={:,}, n_feat={})...'.format(
    len(yama_train_dates) * len(yama_cells), len(yama_cells) + 5))
yama_glm_logit = train_glm_logit(yama_train_L, feat_tr_y, cidx_tr_y,
                                   feat_te_y, cidx_te_y, len(yama_cells))
print(f'  完了  scores shape={yama_glm_logit.shape}  mean={yama_glm_logit.mean():.4f}')

print('[GLM-Logit] 秋田 訓練中  (n_obs={:,}, n_feat={})...'.format(
    len(akita_train_dates) * len(akita_cells), len(akita_cells) + 5))
akita_glm_logit = train_glm_logit(akita_train_L, feat_tr_a, cidx_tr_a,
                                   feat_te_a, cidx_te_a, len(akita_cells))
print(f'  完了  scores shape={akita_glm_logit.shape}  mean={akita_glm_logit.mean():.4f}')

r20_yama_gl = _r20(yama_glm_logit, yama_test_L)
r20_akita_gl = _r20(akita_glm_logit, akita_test_L)
print(f'\n  GLM-Logit Recall@20  Yamagata={r20_yama_gl:.4f}  Akita={r20_akita_gl:.4f}')
print(f'GLM-Logit 完了 ({_tm.time()-_t_glm:.1f}s)')

[GLM-Logit] 山形 訓練中  (n_obs=328,896, n_feat=149)...
  完了  scores shape=(365, 144)  mean=0.0111
[GLM-Logit] 秋田 訓練中  (n_obs=261,560, n_feat=265)...
  完了  scores shape=(365, 260)  mean=0.0380

  GLM-Logit Recall@20  Yamagata=0.5470  Akita=0.4543
GLM-Logit 完了 (8.0s)


In [8]:
# ── Poisson GLM ───────────────────────────────────────────────────────────────

def train_poisson_glm(train_L, feat_tr, cell_idx_tr, feat_te, cell_idx_te,
                       n_cells, alpha=POISSON_ALPHA, mode='fixed'):
    """
    Poisson 回帰でセル別切片 + 共有/cell_season 係数を学習。
    スコアは 1 - exp(-λ) に変換して binary 確率として返す。

    Args:
        mode: 'fixed' = 共有係数, 'cell_season' = セル別季節性
    Returns:
        scores: (T_test, n_cells) float32
    """
    X_tr = build_design_matrix(feat_tr, cell_idx_tr, n_cells, mode=mode)
    X_te = build_design_matrix(feat_te, cell_idx_te, n_cells, mode=mode)
    y_tr = train_L.flatten().astype(np.float64)

    reg = PoissonRegressor(alpha=alpha, fit_intercept=False, max_iter=2000,
                            tol=1e-4, warm_start=False)
    reg.fit(X_tr, y_tr)
    lam = reg.predict(X_te)                      # λ の予測
    score = (1.0 - np.exp(-lam)).astype(np.float32)
    T_te = feat_te.shape[0] // n_cells
    return score.reshape(T_te, n_cells)


_t_pg = _tm.time()
print('[Poisson-GLM] 山形 訓練中...')
yama_pois_glm = train_poisson_glm(yama_train_L, feat_tr_y, cidx_tr_y,
                                    feat_te_y, cidx_te_y, len(yama_cells))
print(f'  完了  mean={yama_pois_glm.mean():.4f}')

print('[Poisson-GLM] 秋田 訓練中...')
akita_pois_glm = train_poisson_glm(akita_train_L, feat_tr_a, cidx_tr_a,
                                    feat_te_a, cidx_te_a, len(akita_cells))
print(f'  完了  mean={akita_pois_glm.mean():.4f}')

print('[Poisson-GLM-cs] 山形 訓練中  (n_feat={})...'.format(len(yama_cells) * 4 + 2))
yama_pois_cs = train_poisson_glm(yama_train_L, feat_tr_y, cidx_tr_y,
                                   feat_te_y, cidx_te_y, len(yama_cells),
                                   alpha=POISSON_ALPHA, mode='cell_season')
print(f'  完了  mean={yama_pois_cs.mean():.4f}')

print('[Poisson-GLM-cs] 秋田 訓練中  (n_feat={})...'.format(len(akita_cells) * 4 + 2))
akita_pois_cs = train_poisson_glm(akita_train_L, feat_tr_a, cidx_tr_a,
                                   feat_te_a, cidx_te_a, len(akita_cells),
                                   alpha=POISSON_ALPHA, mode='cell_season')
print(f'  完了  mean={akita_pois_cs.mean():.4f}')

print(f'\nPoisson-GLM / Poisson-GLM-cs 完了 ({_tm.time()-_t_pg:.1f}s)')
for nm, sc_y, sc_a in [('Poisson-GLM',    yama_pois_glm, akita_pois_glm),
                        ('Poisson-GLM-cs', yama_pois_cs,  akita_pois_cs)]:
    print(f'  {nm} Recall@20  Yama={_r20(sc_y, yama_test_L):.4f}  '
          f'Akita={_r20(sc_a, akita_test_L):.4f}')

[Poisson-GLM] 山形 訓練中...
  完了  mean=0.0118
[Poisson-GLM] 秋田 訓練中...
  完了  mean=0.1142
[Poisson-GLM-cs] 山形 訓練中  (n_feat=578)...
  完了  mean=0.0119
[Poisson-GLM-cs] 秋田 訓練中  (n_feat=1042)...
  完了  mean=0.1146

Poisson-GLM / Poisson-GLM-cs 完了 (3.4s)
  Poisson-GLM Recall@20  Yama=0.0267  Akita=0.0033
  Poisson-GLM-cs Recall@20  Yama=0.0267  Akita=0.0033


---
## Part 3: 階層ベイズ Poisson (PyMC + numpyro)

**モデル**:
```
y_{c,t} ~ Poisson(λ_{c,t})
log(λ_{c,t}) = α_c + β_c·recent30_{c,t} + γ₁·sin(doy) + γ₂·cos(doy) + δ·year_idx
α_c ~ Normal(μ_α, σ_α)   # 階層構造: cell 別切片
β_c ~ Normal(μ_β, σ_β)   # 階層構造: cell 別 recent30 反応性
μ_α, μ_β ~ Normal(0, 2)
σ_α, σ_β ~ HalfNormal(1)
γ₁, γ₂, δ ~ Normal(0, 2)
```

**non-centered parameterization** で divergence を防止:
```
α_c = μ_α + σ_α · α_raw_c,  α_raw_c ~ Normal(0, 1)
```

**計算量削減**: `HIER_TRAIN_YEARS` で直近 N 年の訓練データのみ使用。
`HIER_TRAIN_YEARS=0` で全期間を使用 (CPU で 2-4 時間かかる場合がある)。

In [9]:
try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
    print(f'PyMC {pm.__version__}  arviz {az.__version__}')
except ImportError as _e:
    HAS_PYMC = False
    print(f'PyMC/arviz 未インストール: {_e}')
    print('GLM 結果のみで続行します')


def prepare_hier_data(train_L, train_dates, feat_tr, cell_idx_tr, hier_train_years):
    """
    階層ベイズ用の観測配列を作成。直近 hier_train_years 年に絞る。
    hier_train_years=0 のとき全期間使用。

    Returns:
        y_obs:    (n_obs,)  int32  binary 0/1
        r30:      (n_obs,)  float32
        sin_doy:  (n_obs,)  float32
        cos_doy:  (n_obs,)  float32
        yr_idx:   (n_obs,)  float32
        cell_idx: (n_obs,)  int32
        n_cells:  int
    """
    T_tr, n_cells = train_L.shape

    if hier_train_years > 0:
        cutoff = train_dates[-1] - pd.DateOffset(years=hier_train_years)
        mask   = train_dates >= cutoff
        # 対応するロング形式インデックス
        day_idx_full = np.repeat(np.arange(T_tr, dtype=np.int32), n_cells)
        obs_mask = np.isin(day_idx_full, np.where(mask)[0])
        y_obs    = train_L.flatten().astype(np.int32)[obs_mask]
        r30      = feat_tr[:, 0][obs_mask]
        sin_d    = feat_tr[:, 2][obs_mask]
        cos_d    = feat_tr[:, 3][obs_mask]
        yr_i     = feat_tr[:, 4][obs_mask]
        cidx     = cell_idx_tr[obs_mask]
    else:
        y_obs = train_L.flatten().astype(np.int32)
        r30   = feat_tr[:, 0]
        sin_d = feat_tr[:, 2]
        cos_d = feat_tr[:, 3]
        yr_i  = feat_tr[:, 4]
        cidx  = cell_idx_tr

    print(f'  階層ベイズ学習データ: {len(y_obs):,} obs  '
          f'(sighting_rate={y_obs.mean():.4f}  '
          f'train_years used={hier_train_years if hier_train_years>0 else "all"})')
    return y_obs, r30, sin_d, cos_d, yr_i, cidx, n_cells


def fit_hier_bayes(y_obs, r30, sin_d, cos_d, yr_i, cell_idx, n_cells,
                    draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS,
                    use_mcmc=USE_MCMC, seed=RAND_SEED):
    """
    PyMC 階層ベイズ Poisson モデルをフィット。

    non-centered parameterization を使用してサンプリング効率を改善。

    Returns:
        trace:  arviz.InferenceData
    """
    if not HAS_PYMC:
        raise RuntimeError('PyMC が利用不可。USE_MCMC=False にするか pip install pymc')

    # データを float64 に統一
    r30    = r30.astype(np.float64)
    sin_d  = sin_d.astype(np.float64)
    cos_d  = cos_d.astype(np.float64)
    yr_i   = yr_i.astype(np.float64)

    with pm.Model() as model:
        # Hyper-priors
        mu_alpha  = pm.Normal('mu_alpha',  0.0, 2.0)
        sig_alpha = pm.HalfNormal('sig_alpha', 1.0)
        mu_beta   = pm.Normal('mu_beta',   0.0, 2.0)
        sig_beta  = pm.HalfNormal('sig_beta',  1.0)

        # Non-centered cell params
        alpha_raw = pm.Normal('alpha_raw', 0.0, 1.0, shape=n_cells)
        beta_raw  = pm.Normal('beta_raw',  0.0, 1.0, shape=n_cells)
        alpha     = pm.Deterministic('alpha', mu_alpha  + sig_alpha * alpha_raw)
        beta      = pm.Deterministic('beta',  mu_beta   + sig_beta  * beta_raw)

        # Global seasonality + trend
        gamma_sin = pm.Normal('gamma_sin', 0.0, 2.0)
        gamma_cos = pm.Normal('gamma_cos', 0.0, 2.0)
        delta     = pm.Normal('delta',     0.0, 2.0)

        # Log-linear predictor
        eta = (alpha[cell_idx] + beta[cell_idx] * r30
               + gamma_sin * sin_d + gamma_cos * cos_d
               + delta * yr_i)
        lam = pm.math.exp(eta)

        # Likelihood
        pm.Poisson('y_obs', mu=lam, observed=y_obs)

        if use_mcmc:
            trace = pm.sample(
                draws=draws, tune=tune, chains=chains,
                nuts_sampler='numpyro',
                target_accept=0.9,
                random_seed=seed,
                progressbar=True,
            )
        else:
            approx = pm.fit(n=10000, method='fullrank_advi',
                            random_seed=seed, progressbar=True)
            trace  = approx.sample(draws * chains)

    return trace, model


def hier_scores_from_trace(trace, model, feat_te, cell_idx_te, n_cells):
    """
    事後分布サンプルから P(Y >= 1) = 1 - P(Y = 0) = 1 - exp(-E[λ]) を計算。

    Returns:
        scores: (T_test, n_cells) float32
    """
    # 事後平均パラメータで λ を計算
    alpha_mean = trace.posterior['alpha'].values.mean(axis=(0, 1))  # (n_cells,)
    beta_mean  = trace.posterior['beta'].values.mean(axis=(0, 1))
    gs_mean    = float(trace.posterior['gamma_sin'].values.mean())
    gc_mean    = float(trace.posterior['gamma_cos'].values.mean())
    d_mean     = float(trace.posterior['delta'].values.mean())

    r30    = feat_te[:, 0].astype(np.float64)
    sin_d  = feat_te[:, 2].astype(np.float64)
    cos_d  = feat_te[:, 3].astype(np.float64)
    yr_i   = feat_te[:, 4].astype(np.float64)

    eta  = (alpha_mean[cell_idx_te] + beta_mean[cell_idx_te] * r30
            + gs_mean * sin_d + gc_mean * cos_d + d_mean * yr_i)
    lam  = np.exp(eta)
    score = (1.0 - np.exp(-lam)).astype(np.float32)
    T_te = feat_te.shape[0] // n_cells
    return score.reshape(T_te, n_cells)


def hier_uncertainty_scores(trace, feat_te, cell_idx_te, n_cells):
    """
    事後サンプル全てから P(Y>=1) を計算し、各 (cell,day) の事後平均・std を返す。

    Returns:
        mean_scores: (T_test, n_cells) float32  事後平均スコア
        std_scores:  (T_test, n_cells) float32  事後標準偏差
    """
    n_post = trace.posterior.dims['draw'] * trace.posterior.dims['chain']
    alpha_s = trace.posterior['alpha'].values.reshape(n_post, n_cells)   # (S, n_cells)
    beta_s  = trace.posterior['beta'].values.reshape(n_post, n_cells)
    gs_s    = trace.posterior['gamma_sin'].values.flatten()              # (S,)
    gc_s    = trace.posterior['gamma_cos'].values.flatten()
    d_s     = trace.posterior['delta'].values.flatten()

    r30_   = feat_te[:, 0].astype(np.float64)
    sin_d_ = feat_te[:, 2].astype(np.float64)
    cos_d_ = feat_te[:, 3].astype(np.float64)
    yr_i_  = feat_te[:, 4].astype(np.float64)

    T_te = feat_te.shape[0] // n_cells
    all_scores = np.empty((n_post, T_te * n_cells), dtype=np.float32)
    for i in range(n_post):
        eta = (alpha_s[i, cell_idx_te] + beta_s[i, cell_idx_te] * r30_
               + gs_s[i] * sin_d_ + gc_s[i] * cos_d_ + d_s[i] * yr_i_)
        all_scores[i] = (1.0 - np.exp(-np.exp(eta))).astype(np.float32)

    mean_sc = all_scores.mean(axis=0).reshape(T_te, n_cells)
    std_sc  = all_scores.std(axis=0).reshape(T_te, n_cells)
    return mean_sc, std_sc


print('PyMC モデル関数定義 OK')
print(f'HAS_PYMC={HAS_PYMC}  USE_MCMC={USE_MCMC}  HIER_TRAIN_YEARS={HIER_TRAIN_YEARS}')

PyMC 5.28.4  arviz 0.22.0
PyMC モデル関数定義 OK
HAS_PYMC=True  USE_MCMC=True  HIER_TRAIN_YEARS=3


In [10]:
_t_hier_y = _tm.time()

if HAS_PYMC:
    print('=' * 60)
    print('[階層ベイズ] 山形 サンプリング開始')
    print('=' * 60)

    yama_hb_data = prepare_hier_data(
        yama_train_L, yama_train_dates, feat_tr_y, cidx_tr_y, HIER_TRAIN_YEARS)
    y_obs_y, r30_y, sin_y, cos_y, yr_y, cidx_hb_y, n_cells_y = yama_hb_data

    yama_trace, yama_hb_model = fit_hier_bayes(
        y_obs_y, r30_y, sin_y, cos_y, yr_y, cidx_hb_y, n_cells_y)

    # ── 収束診断 ──────────────────────────────────────────────────────────────
    print('\n[収束診断] 山形')
    summary = az.summary(yama_trace, var_names=['mu_alpha', 'sig_alpha',
                                                  'mu_beta', 'sig_beta',
                                                  'gamma_sin', 'gamma_cos', 'delta'])
    print(summary[['mean', 'sd', 'r_hat', 'ess_bulk']].to_string())

    rhat_vals = az.rhat(yama_trace, var_names=['alpha_raw', 'beta_raw',
                                                'gamma_sin', 'gamma_cos', 'delta'])
    max_rhat  = float(max(v.values.max() for v in rhat_vals.data_vars.values()))
    n_div = int(yama_trace.sample_stats.diverging.values.sum())
    print(f'\n  max R-hat = {max_rhat:.4f}  (目標 < 1.01)')
    print(f'  Divergent transitions = {n_div}  (目標 < {MCMC_DRAWS * MCMC_CHAINS * 0.01:.0f})')
    if max_rhat > 1.05:
        print('  WARNING: R-hat > 1.05 — 収束不十分。MCMC_TUNE を増やすかモデルを確認')
    elif max_rhat > 1.01:
        print('  NOTE: R-hat 1.01-1.05 — 許容範囲だが注意')
    else:
        print('  収束 OK')

    # ── スコア生成 ────────────────────────────────────────────────────────────
    print('\n事後予測スコア生成 (山形)...')
    yama_hier_scores = hier_scores_from_trace(
        yama_trace, yama_hb_model, feat_te_y, cidx_te_y, n_cells_y)
    yama_hier_mean, yama_hier_std = hier_uncertainty_scores(
        yama_trace, feat_te_y, cidx_te_y, n_cells_y)
    print(f'  scores shape={yama_hier_scores.shape}  mean={yama_hier_scores.mean():.4f}')
    print(f'  R@20 = {_r20(yama_hier_scores, yama_test_L):.4f}')
    print(f'山形 階層ベイズ完了 ({_tm.time()-_t_hier_y:.1f}s)')

else:
    print('PyMC 未インストール — 山形 階層ベイズをスキップ')
    yama_trace = None
    # フォールバック: B5 スコアをコピーして形式を整える
    yama_hier_scores = yama_baselines['B5: B2+B3'].copy()
    yama_hier_mean   = yama_hier_scores.copy()
    yama_hier_std    = np.zeros_like(yama_hier_scores)
    print('  (フォールバック: yama_hier_scores = B5 スコアをコピー)')

[階層ベイズ] 山形 サンプリング開始
  階層ベイズ学習データ: 157,968 obs  (sighting_rate=0.0091  train_years used=3)


  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



[収束診断] 山形
            mean     sd  r_hat  ess_bulk
mu_alpha  -6.267  0.259    1.0     277.0
sig_alpha  1.864  0.166    1.0     256.0
mu_beta    0.307  0.030    1.0     447.0
sig_beta   0.147  0.028    1.0     513.0
gamma_sin -0.382  0.045    1.0    1958.0
gamma_cos -1.050  0.052    1.0    2121.0
delta     -0.060  0.035    1.0    1413.0

  max R-hat = 1.0095  (目標 < 1.01)
  Divergent transitions = 0  (目標 < 20)
  収束 OK

事後予測スコア生成 (山形)...
  scores shape=(365, 144)  mean=0.0238
  R@20 = 0.5453
山形 階層ベイズ完了 (1779.8s)


In [11]:
_t_hier_a = _tm.time()

if HAS_PYMC:
    print('=' * 60)
    print('[階層ベイズ] 秋田 サンプリング開始')
    print('=' * 60)

    akita_hb_data = prepare_hier_data(
        akita_train_L, akita_train_dates, feat_tr_a, cidx_tr_a, HIER_TRAIN_YEARS)
    y_obs_a, r30_a, sin_a, cos_a, yr_a, cidx_hb_a, n_cells_a = akita_hb_data

    akita_trace, akita_hb_model = fit_hier_bayes(
        y_obs_a, r30_a, sin_a, cos_a, yr_a, cidx_hb_a, n_cells_a)

    print('\n[収束診断] 秋田')
    rhat_a = az.rhat(akita_trace, var_names=['alpha_raw', 'beta_raw',
                                              'gamma_sin', 'gamma_cos', 'delta'])
    max_rhat_a = float(max(v.values.max() for v in rhat_a.data_vars.values()))
    n_div_a    = int(akita_trace.sample_stats.diverging.values.sum())
    print(f'  max R-hat = {max_rhat_a:.4f}  Divergences = {n_div_a}')
    if max_rhat_a > 1.01:
        print(f'  WARNING: R-hat = {max_rhat_a:.4f} > 1.01')
    else:
        print('  収束 OK')

    print('\n事後予測スコア生成 (秋田)...')
    akita_hier_scores = hier_scores_from_trace(
        akita_trace, akita_hb_model, feat_te_a, cidx_te_a, n_cells_a)
    akita_hier_mean, akita_hier_std = hier_uncertainty_scores(
        akita_trace, feat_te_a, cidx_te_a, n_cells_a)
    print(f'  scores shape={akita_hier_scores.shape}  mean={akita_hier_scores.mean():.4f}')
    print(f'  R@20 = {_r20(akita_hier_scores, akita_test_L):.4f}')
    print(f'秋田 階層ベイズ完了 ({_tm.time()-_t_hier_a:.1f}s)')

else:
    print('PyMC 未インストール — 秋田 階層ベイズをスキップ')
    akita_trace = None
    akita_hier_scores = akita_baselines['B5: B2+B3'].copy()
    akita_hier_mean   = akita_hier_scores.copy()
    akita_hier_std    = np.zeros_like(akita_hier_scores)
    print('  (フォールバック: akita_hier_scores = B5 スコアをコピー)')

# ── 中間結果保存 ──────────────────────────────────────────────────────────────
np.save(OUTPUT_DIR + '/scores/yama_glm_logit.npy',  yama_glm_logit)
np.save(OUTPUT_DIR + '/scores/yama_pois_glm.npy',   yama_pois_glm)
np.save(OUTPUT_DIR + '/scores/yama_pois_cs.npy',    yama_pois_cs)
np.save(OUTPUT_DIR + '/scores/yama_hier_mean.npy',  yama_hier_mean)
np.save(OUTPUT_DIR + '/scores/akita_glm_logit.npy', akita_glm_logit)
np.save(OUTPUT_DIR + '/scores/akita_pois_glm.npy',  akita_pois_glm)
np.save(OUTPUT_DIR + '/scores/akita_pois_cs.npy',   akita_pois_cs)
np.save(OUTPUT_DIR + '/scores/akita_hier_mean.npy', akita_hier_mean)

if yama_trace is not None:
    import pickle
    with open(OUTPUT_DIR + '/traces/yamagata_hier_trace.pkl', 'wb') as _f:
        pickle.dump({'trace': yama_trace, 'scores': yama_hier_scores}, _f)
    with open(OUTPUT_DIR + '/traces/akita_hier_trace.pkl', 'wb') as _f:
        pickle.dump({'trace': akita_trace, 'scores': akita_hier_scores}, _f)
    print('\nTraces 保存完了')

print('\n全スコア保存完了:', OUTPUT_DIR + '/scores/')

[階層ベイズ] 秋田 サンプリング開始
  階層ベイズ学習データ: 261,560 obs  (sighting_rate=0.0181  train_years used=3)


  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



[収束診断] 秋田
  max R-hat = 1.0145  Divergences = 0

事後予測スコア生成 (秋田)...
  scores shape=(365, 260)  mean=0.0527
  R@20 = 0.4313
秋田 階層ベイズ完了 (4730.9s)

Traces 保存完了

全スコア保存完了: /content/drive/MyDrive/ttm_bear_glm_results/scores/


---
## Part 4: 全手法統合比較

全 12 手法 × 両県 × 全メトリクスの結果テーブルを生成する。

In [12]:
_t_eval = _tm.time()

# ── 全手法スコア辞書 ─────────────────────────────────────────────────────────
yama_all_scores = {
    **yama_baselines,
    'ET':               yama_et_scores,
    'TTM':              yama_ttm_scores,
    'GLM-Logit':        yama_glm_logit,
    'Poisson-GLM':      yama_pois_glm,
    'Poisson-GLM-cs':   yama_pois_cs,
    'HierBayes':        yama_hier_scores,
}
akita_all_scores = {
    **akita_baselines,
    'ET':               akita_et_scores,
    'TTM':              akita_ttm_scores,
    'GLM-Logit':        akita_glm_logit,
    'Poisson-GLM':      akita_pois_glm,
    'Poisson-GLM-cs':   akita_pois_cs,
    'HierBayes':        akita_hier_scores,
}

METHOD_ORDER = ['B0: Random', 'B1: Static prior', 'B2: Recent MA', 'B3: DoY season',
                'B4: B1+B3', 'B5: B2+B3', 'ET', 'TTM',
                'GLM-Logit', 'Poisson-GLM', 'Poisson-GLM-cs', 'HierBayes']


def build_results_df(scores_dict, test_L, methods=METHOD_ORDER):
    rows = []
    for name in methods:
        if name not in scores_dict:
            continue
        sc = scores_dict[name]
        r = evaluate_all(sc, test_L)
        r['Method'] = name
        rows.append(r)
    df = pd.DataFrame(rows).set_index('Method')
    main_cols = (['Recall@10', 'Recall@20', 'Recall@30',
                  'Prec@10', 'Prec@20', 'Prec@30',
                  'ROC-AUC', 'PR-AUC', 'Brier', 'BSS', 'ECE', 'MAE'])
    return df[[c for c in main_cols if c in df.columns]]


print('全手法評価中...')
yama_res = build_results_df(yama_all_scores, yama_test_L)
akita_res = build_results_df(akita_all_scores, akita_test_L)

fmt = '{:.4f}'.format
print('\n=== Yamagata — 全手法結果 ===')
with pd.option_context('display.float_format', fmt, 'display.width', 160,
                        'display.max_columns', 20):
    print(yama_res.to_string())

print('\n=== Akita — 全手法結果 ===')
with pd.option_context('display.float_format', fmt, 'display.width', 160,
                        'display.max_columns', 20):
    print(akita_res.to_string())

print(f'\n評価完了 ({_tm.time()-_t_eval:.1f}s)')

全手法評価中...

=== Yamagata — 全手法結果 ===
                  Recall@10  Recall@20  Recall@30  Prec@10  Prec@20  Prec@30  ROC-AUC  PR-AUC  Brier     BSS    ECE    MAE
Method                                                                                                                    
B0: Random           0.0602     0.1256     0.1860   0.0624   0.0615   0.0612   0.5022  0.0720 0.3343 -8.0621 0.4625 0.5011
B1: Static prior     0.2864     0.5331     0.6588   0.2441   0.2319   0.1973   0.5000  0.0614 0.0366  0.0070 0.0292 0.0454
B2: Recent MA        0.3106     0.4857     0.6066   0.2925   0.2387   0.2019   0.6932  0.1368 0.0314  0.1478 0.0092 0.0595
B3: DoY season       0.3048     0.4749     0.5871   0.2573   0.2136   0.1862   0.6631  0.1206 0.0355  0.0381 0.0292 0.0439
B4: B1+B3            0.3204     0.5171     0.6441   0.2671   0.2254   0.1961   0.4994  0.0897 0.1445 -2.9168 0.1561 0.1700
B5: B2+B3            0.3326     0.5342     0.6604   0.3005   0.2437   0.2086   0.5497  0.1003 0.1253 -2

In [13]:
from tqdm.notebook import tqdm

_t_boot = _tm.time()

KEY_COMPARISONS = [
    ('GLM-Logit',      'TTM'),
    ('GLM-Logit',      'ET'),
    ('GLM-Logit',      'B1: Static prior'),
    ('GLM-Logit',      'B5: B2+B3'),
    ('Poisson-GLM',    'TTM'),
    ('Poisson-GLM',    'GLM-Logit'),
    ('Poisson-GLM-cs', 'B5: B2+B3'),
    ('Poisson-GLM-cs', 'Poisson-GLM'),
    ('HierBayes',      'TTM'),
    ('HierBayes',      'ET'),
    ('HierBayes',      'GLM-Logit'),
    ('HierBayes',      'Poisson-GLM'),
    ('HierBayes',      'B5: B2+B3'),
]

N_COMPARISONS = len(KEY_COMPARISONS) * len(K_VALUES) * 2   # day + perm
ALPHA_BONF = 0.05 / (len(KEY_COMPARISONS) * len(K_VALUES) * 3)  # 3 test types


def day_bootstrap(sA, sB, labels, K, B=B_BOOT, seed=RAND_SEED):
    rng = np.random.default_rng(seed)
    RA, valid = _per_day_recall(sA, labels, K)
    RB, _     = _per_day_recall(sB, labels, K)
    diff = (RA - RB)[valid]; n_v = int(valid.sum()); obs = float(diff.mean())
    idx  = rng.integers(0, n_v, size=(B, n_v))
    boot = diff[idx].mean(axis=1)
    ci   = tuple(np.percentile(boot, [2.5, 97.5]))
    pval = float((np.abs(boot - obs) >= np.abs(obs)).sum() + 1) / (B + 1)
    return obs, ci, pval, n_v


def day_permutation(sA, sB, labels, K, P=B_PERM, seed=RAND_SEED):
    rng = np.random.default_rng(seed)
    RA, valid = _per_day_recall(sA, labels, K)
    RB, _     = _per_day_recall(sB, labels, K)
    diff = (RA - RB)[valid]; obs = float(diff.mean())
    signs = rng.choice([-1., 1.], size=(P, int(valid.sum())))
    perm  = (signs * diff).mean(axis=1)
    return obs, float((np.abs(perm) >= np.abs(obs)).sum() + 1) / (P + 1)


def run_comparisons(pref, scores_dict, test_L, comparisons, k_values):
    rows = []
    total = len(comparisons) * len(k_values)
    with tqdm(total=total, desc=f'{pref} bootstrap') as pbar:
        for nameA, nameB in comparisons:
            if nameA not in scores_dict or nameB not in scores_dict:
                pbar.update(len(k_values))
                continue
            sA, sB = scores_dict[nameA], scores_dict[nameB]
            for K in k_values:
                obs_b, ci, pv_b, nv = day_bootstrap(sA, sB, test_L, K)
                obs_p, pv_p         = day_permutation(sA, sB, test_L, K)
                sig_b  = '***' if (ci[0] > 0 or ci[1] < 0) else 'ns'
                sig_p  = '*' if pv_p < 0.05 else 'ns'
                sig_bf = 'sig*' if pv_b < ALPHA_BONF else ('ns')
                rows.append({
                    'Comparison': f'{nameA} vs {nameB}', 'K': K,
                    'obs': obs_b, 'ci_lo': ci[0], 'ci_hi': ci[1],
                    'p_boot': pv_b, 'p_perm': pv_p,
                    'sig_boot': sig_b, 'sig_perm': sig_p,
                    'sig_bonf': sig_bf, 'n_valid': nv,
                })
                pbar.update(1)
    return pd.DataFrame(rows)


print('Bootstrap + Permutation test 実行中...')
yama_boot_df = run_comparisons('Yamagata', yama_all_scores, yama_test_L,
                                 KEY_COMPARISONS, K_VALUES)
akita_boot_df = run_comparisons('Akita',   akita_all_scores, akita_test_L,
                                 KEY_COMPARISONS, K_VALUES)


def print_boot_summary(pref, df, K=20):
    print(f'\n=== {pref} Bootstrap CI (K={K}) ===')
    sub = df[df['K'] == K]
    print(f'  Bonferroni α = {ALPHA_BONF:.6f}  ({N_COMPARISONS} total tests)')
    for _, r in sub.iterrows():
        print(f'  {r["Comparison"]:35s}  obs={r["obs"]:+.4f}  '
              f'CI=[{r["ci_lo"]:+.4f},{r["ci_hi"]:+.4f}]  '
              f'p_boot={r["p_boot"]:.4f}  p_perm={r["p_perm"]:.4f}  '
              f'boot={r["sig_boot"]}  bonf={r["sig_bonf"]}')


print_boot_summary('Yamagata', yama_boot_df)
print_boot_summary('Akita',    akita_boot_df)
print(f'\nBootstrap 完了 ({_tm.time()-_t_boot:.1f}s)')

Bootstrap + Permutation test 実行中...


Yamagata bootstrap:   0%|          | 0/39 [00:00<?, ?it/s]

Akita bootstrap:   0%|          | 0/39 [00:00<?, ?it/s]


=== Yamagata Bootstrap CI (K=20) ===
  Bonferroni α = 0.000427  (78 total tests)
  GLM-Logit vs TTM                     obs=+0.0554  CI=[+0.0326,+0.0792]  p_boot=0.0002  p_perm=0.0002  boot=***  bonf=sig*
  GLM-Logit vs ET                      obs=+0.0731  CI=[+0.0493,+0.0975]  p_boot=0.0002  p_perm=0.0002  boot=***  bonf=sig*
  GLM-Logit vs B1: Static prior        obs=+0.0139  CI=[-0.0062,+0.0322]  p_boot=0.1528  p_perm=0.1576  boot=ns  bonf=ns
  GLM-Logit vs B5: B2+B3               obs=+0.0129  CI=[-0.0129,+0.0404]  p_boot=0.3469  p_perm=0.3535  boot=ns  bonf=ns
  Poisson-GLM vs TTM                   obs=-0.4649  CI=[-0.4975,-0.4322]  p_boot=0.0002  p_perm=0.0002  boot=***  bonf=sig*
  Poisson-GLM vs GLM-Logit             obs=-0.5203  CI=[-0.5530,-0.4863]  p_boot=0.0002  p_perm=0.0002  boot=***  bonf=sig*
  Poisson-GLM-cs vs B5: B2+B3          obs=-0.5075  CI=[-0.5398,-0.4736]  p_boot=0.0002  p_perm=0.0002  boot=***  bonf=sig*
  Poisson-GLM-cs vs Poisson-GLM        obs=+0.0000  CI=[

---
## Part 5: 階層ベイズの差別化分析

1. **Partial pooling の可視化**: データ量 vs α_c 事後分布
2. **少データ cell での性能比較**: Low/Medium/High 層別 Recall@20
3. **不確実性定量化**: 予測確信度別の Recall@20

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if yama_trace is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, pref, trace, train_L, cells_list in [
        (axes[0], 'Yamagata', yama_trace, yama_train_L, yama_cells),
        (axes[1], 'Akita',    akita_trace, akita_train_L, akita_cells),
    ]:
        n_cells = train_L.shape[1]
        train_count = train_L.sum(axis=0)   # (n_cells,) 訓練期間の目撃数

        alpha_post  = trace.posterior['alpha'].values.reshape(-1, n_cells)
        alpha_mean  = alpha_post.mean(axis=0)
        alpha_ci_lo = np.percentile(alpha_post, 2.5,  axis=0)
        alpha_ci_hi = np.percentile(alpha_post, 97.5, axis=0)
        alpha_width = alpha_ci_hi - alpha_ci_lo

        sc = ax.scatter(train_count, alpha_mean, c=alpha_width,
                        cmap='plasma', alpha=0.7, s=30)
        plt.colorbar(sc, ax=ax, label='95% CI width')
        ax.set_xlabel('Training sightings (count)', fontsize=11)
        ax.set_ylabel('Posterior mean alpha_c', fontsize=11)
        ax.set_title(pref + ': Partial Pooling (color=CI width)', fontsize=11)

        # グローバル平均線
        mu_alpha_mean = float(trace.posterior['mu_alpha'].values.mean())
        ax.axhline(mu_alpha_mean, color='red', linestyle='--',
                    alpha=0.7, label='mu_alpha = %.3f' % mu_alpha_mean)
        ax.legend(fontsize=9)

    plt.tight_layout()
    pp_path = OUTPUT_DIR + '/partial_pooling_visualization.png'
    plt.savefig(pp_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'保存: {pp_path}')
else:
    print('PyMC trace なし — 可視化をスキップ')
    print('(yama_trace = None のため partial pooling 可視化不可)')

保存: /content/drive/MyDrive/ttm_bear_glm_results/partial_pooling_visualization.png


In [15]:
# ── 少データ cell での性能比較 (Low/Medium/High 層別 Recall@20) ───────────────────

def stratified_recall(scores_dict, train_L, test_L, K=20, methods=None):
    """
    訓練期間の目撃数で cells を Low/Medium/High に3分割し、
    各層内の Recall@K を計算する。

    Returns:
        df: MultiIndex (Method, Stratum) → Recall@K
    """
    if methods is None:
        methods = list(scores_dict.keys())
    n_cells = train_L.shape[1]
    count   = train_L.sum(axis=0)
    thirds  = np.quantile(count, [1/3, 2/3])
    strata  = {
        'Low':    np.where(count <= thirds[0])[0],
        'Medium': np.where((count > thirds[0]) & (count <= thirds[1]))[0],
        'High':   np.where(count > thirds[1])[0],
    }
    rows = []
    for name in methods:
        if name not in scores_dict:
            continue
        sc = scores_dict[name]
        for stratum, idx in strata.items():
            K_s = min(K, len(idx) - 1) if len(idx) > K else len(idx)
            if K_s < 1:
                val = float('nan')
            else:
                s_s = sc[:, idx]; l_s = test_L[:, idx].astype(np.int32)
                topk = np.argpartition(-s_s, K_s, axis=1)[:, :K_s]
                hits = np.take_along_axis(l_s, topk, axis=1).sum(axis=1)
                npos = l_s.sum(axis=1).astype(np.float64)
                valid = npos > 0
                val = float((hits[valid] / npos[valid]).mean()) if valid.any() else 0.0
            rows.append({'Method': name, 'Stratum': stratum, f'Recall@{K}': val,
                          'n_cells': len(idx), 'mean_count': float(count[idx].mean())})
    return pd.DataFrame(rows).pivot(index='Method', columns='Stratum', values=f'Recall@{K}')


focus_methods = ['B1: Static prior', 'B2: Recent MA', 'B5: B2+B3',
                  'ET', 'TTM', 'GLM-Logit', 'Poisson-GLM', 'HierBayes']

yama_strat = stratified_recall(yama_all_scores, yama_train_L, yama_test_L,
                                 K=20, methods=focus_methods)
akita_strat = stratified_recall(akita_all_scores, akita_train_L, akita_test_L,
                                 K=20, methods=focus_methods)

fmt = '{:.4f}'.format
print('=== Yamagata 層別 Recall@20 ===')
with pd.option_context('display.float_format', fmt, 'display.width', 80):
    print(yama_strat[['Low', 'Medium', 'High']].to_string())

print('\n=== Akita 層別 Recall@20 ===')
with pd.option_context('display.float_format', fmt, 'display.width', 80):
    print(akita_strat[['Low', 'Medium', 'High']].to_string())

=== Yamagata 層別 Recall@20 ===
Stratum             Low  Medium   High
Method                                
B1: Static prior 1.0000  0.5992 0.6170
B2: Recent MA    0.6667  0.7002 0.6002
B5: B2+B3        1.0000  0.6954 0.6372
ET               1.0000  0.4523 0.5569
GLM-Logit        1.0000  0.6977 0.6355
HierBayes        0.6667  0.6561 0.6363
Poisson-GLM      0.6667  0.2546 0.2288
TTM              0.6667  0.5615 0.5727

=== Akita 層別 Recall@20 ===
Stratum             Low  Medium   High
Method                                
B1: Static prior 0.0000  0.7654 0.4413
B2: Recent MA    0.0000  0.8100 0.4675
B5: B2+B3        0.0000  0.8243 0.4795
ET               0.4167  0.7246 0.3555
GLM-Logit        0.0000  0.7856 0.4948
HierBayes        0.0000  0.7799 0.4705
Poisson-GLM      0.0000  0.1942 0.0736
TTM              0.0000  0.5947 0.4294


In [16]:
# ── 不確実性定量化: 予測確信度別 Recall@20 ────────────────────────────────────────
# 「確信度が高い予測のみで評価した場合の Recall@K」
# 確信度 = 事後標準偏差が低い (std が低い = 確信度が高い)

def confidence_stratified_recall(mean_sc, std_sc, labels, K=20,
                                   conf_thresholds=(0.25, 0.50, 0.75, 1.0)):
    """
    各 (cell, day) の確信度を std の逆数で評価。
    上位 p% の確信度 (低 std) を持つ (cell,day) ペアのみで Recall@K を計算。

    conf_thresholds: 使用する (cell,day) の割合 (最も確信度が高い上位 p%)
    Returns:
        dict: {p: recall_at_k}
    """
    n_days, n_cells = labels.shape
    # 確信度スコア (高いほど確信: 1/(std+ε))
    conf_score = 1.0 / (std_sc + 1e-6)

    results = {}
    for p in conf_thresholds:
        if p >= 1.0:
            # 全データ使用
            r = global_recall_at_k(mean_sc, labels, K)
        else:
            # 上位 p の確信度を持つ日×セルを「マスクして」評価
            threshold = np.quantile(conf_score, 1.0 - p)
            high_conf = conf_score >= threshold  # (T, n_cells) bool
            # 各日の upper-K を high_conf セルの中から選ぶ
            hits_list, npos_list = [], []
            for t in range(n_days):
                hc_idx = np.where(high_conf[t])[0]
                if len(hc_idx) < K + 1:
                    continue
                s_hc = mean_sc[t, hc_idx]
                l_hc = labels[t, hc_idx].astype(np.int32)
                K_eff = min(K, len(hc_idx) - 1)
                topk  = np.argpartition(-s_hc, K_eff)[:K_eff]
                hits_list.append(l_hc[topk].sum())
                npos_list.append(l_hc.sum())
            if npos_list and sum(1 for x in npos_list if x > 0) > 0:
                valid_pairs = [(h, n) for h, n in zip(hits_list, npos_list) if n > 0]
                r = float(np.mean([h / n for h, n in valid_pairs]))
            else:
                r = float('nan')
        results[f'top{int(p*100)}%'] = r
    return results


if yama_trace is not None and yama_hier_std is not None:
    print('=== 不確実性定量化: 階層ベイズ予測の確信度別 Recall@20 ===')
    for pref, mean_sc, std_sc, labels in [
        ('Yamagata', yama_hier_mean, yama_hier_std, yama_test_L),
        ('Akita',    akita_hier_mean, akita_hier_std, akita_test_L),
    ]:
        unc_res = confidence_stratified_recall(mean_sc, std_sc, labels, K=20)
        print(f'  [{pref}]')
        for k, v in unc_res.items():
            print(f'    {k}: Recall@20 = {v:.4f}')
    print()
    print('解釈: top25% (最も確信度が高い予測のみ) の Recall@20 が全体より高ければ、')
    print('  階層ベイズの不確実性は識別力を持つ (高確信 = 高精度)。')
else:
    print('PyMC trace なし — 不確実性定量化をスキップ')
    print('HierBayes 事後標準偏差が利用不可です (USE_MCMC=True かつ PyMC インストール必要)')

=== 不確実性定量化: 階層ベイズ予測の確信度別 Recall@20 ===
  [Yamagata]
    top25%: Recall@20 = 1.0000
    top50%: Recall@20 = 0.8438
    top75%: Recall@20 = 0.7497
    top100%: Recall@20 = 0.5425
  [Akita]
    top25%: Recall@20 = 1.0000
    top50%: Recall@20 = 0.7143
    top75%: Recall@20 = 0.6628
    top100%: Recall@20 = 0.4316

解釈: top25% (最も確信度が高い予測のみ) の Recall@20 が全体より高ければ、
  階層ベイズの不確実性は識別力を持つ (高確信 = 高精度)。


In [17]:
# ── Forest plot: Recall@20 + Brier (CI 付き) ─────────────────────────────────────

def compute_bootstrap_ci(sc, labels, metric_fn, B=500, seed=RAND_SEED):
    """
    Bootstrap で各手法の指標の95% CI を推定する。
    metric_fn: (scores, labels) -> float
    """
    rng = np.random.default_rng(seed)
    n = len(labels)
    vals = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        vals.append(metric_fn(sc[idx], labels[idx]))
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def metric_r20(sc, lab): return _r20(sc, lab)
def metric_brier(sc, lab): return float(np.mean((np.clip(sc, 0, 1) - lab) ** 2))


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
methods_plot = [m for m in METHOD_ORDER if m in yama_all_scores]
colors = plt.cm.tab20(np.linspace(0, 1, len(methods_plot)))

for row, (metric_fn, metric_name) in enumerate([(metric_r20, 'Recall@20'),
                                                   (metric_brier, 'Brier')]):
    for col, (pref, scores_dict, test_L) in enumerate([
        ('Yamagata', yama_all_scores, yama_test_L),
        ('Akita',    akita_all_scores, akita_test_L)
    ]):
        ax = axes[row][col]
        means, lo_list, hi_list, names = [], [], [], []
        for name in methods_plot:
            if name not in scores_dict:
                continue
            m, lo, hi = compute_bootstrap_ci(scores_dict[name], test_L, metric_fn, B=500)
            means.append(m); lo_list.append(lo); hi_list.append(hi)
            names.append(name)

        y_pos = np.arange(len(names))
        ax.barh(y_pos, means, xerr=[np.array(means) - np.array(lo_list),
                                     np.array(hi_list) - np.array(means)],
                color=colors[:len(names)], alpha=0.75, capsize=4, height=0.6)
        ax.set_yticks(y_pos); ax.set_yticklabels(names, fontsize=9)
        ax.set_xlabel(metric_name, fontsize=11)
        ax.set_title(f'{pref} — {metric_name} (95% CI, B=500)', fontsize=12)
        ax.invert_yaxis()
        if metric_name == 'Recall@20':
            ax.axvline(0.492 if pref == 'Yamagata' else 0.395,
                        color='red', linestyle=':', alpha=0.5, label='Paper TTM')
        ax.legend(fontsize=8)
        ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
fp_path = OUTPUT_DIR + '/forest_plot_combined.png'
plt.savefig(fp_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Forest plot 保存: {fp_path}')

Forest plot 保存: /content/drive/MyDrive/ttm_bear_glm_results/forest_plot_combined.png


---
## Part 6: 自動判定と論文への含意

シナリオ自動判定:
- **Y**: 階層ベイズ > 全手法 → 「HierBayes が稀少地理空間事象予測の最適解」
- **X**: GLM > TTM/ET → 「Foundation model は古典統計を超えていない」
- **Z**: TTM > GLM/HierBayes → 元の主張を強化
- **W**: B2/B5 > 全学習手法 → 「なぜ naive rolling average が最強か」

In [18]:
def auto_verdict(yama_res, akita_res, boot_df_y, boot_df_a):
    """
    全手法 Recall@20 と Brier をもとに、論文の進路を自動判定する。
    """
    def get(df, method, col='Recall@20'):
        return float(df.loc[method, col]) if method in df.index else float('nan')

    yr20  = {m: get(yama_res, m) for m in METHOD_ORDER if m in yama_res.index}
    ar20  = {m: get(akita_res, m) for m in METHOD_ORDER if m in akita_res.index}
    yb    = {m: get(yama_res, m, 'Brier') for m in yr20}

    ttm_y = yr20.get('TTM', 0); et_y = yr20.get('ET', 0)
    gl_y  = yr20.get('GLM-Logit', 0); pg_y = yr20.get('Poisson-GLM', 0)
    pc_y  = yr20.get('Poisson-GLM-cs', 0); hb_y = yr20.get('HierBayes', 0)
    b5_y  = yr20.get('B5: B2+B3', 0);  b2_y = yr20.get('B2: Recent MA', 0)

    best_ml   = max(ttm_y, et_y)
    best_glm  = max(gl_y, pg_y, pc_y)
    best_stat = max(best_glm, hb_y)
    best_naive = max(b2_y, b5_y)

    print('=' * 70)
    print('シナリオ自動判定')
    print('=' * 70)
    print(f'  [Yamagata Recall@20]')
    for name, val in sorted(yr20.items(), key=lambda x: -x[1]):
        print(f'    {name:25s}: {val:.4f}')

    print()
    if hb_y > max(best_ml, best_glm, best_naive) and hb_y > 0:
        scenario = 'Y'
        print('  ★ シナリオ Y: 階層ベイズが全手法を上回る')
        print('  推奨論文軸: 「HierBayes が稀少地理空間事象予測の最適解」')
        print('  投稿候補 : SIGSPATIAL Research Track / KDD ADS / ECML-PKDD')
    elif best_glm > best_ml and best_glm > best_naive:
        scenario = 'X'
        print('  ★ シナリオ X: GLM が機械学習手法 (TTM/ET) を上回る')
        print('  推奨論文軸: 「Foundation model は古典統計を超えていない」')
        print('  投稿候補 : SIGSPATIAL Vision Track / NeurIPS Workshop')
    elif ttm_y > best_stat and ttm_y > best_naive:
        scenario = 'Z'
        print('  ★ シナリオ Z: TTM が古典統計手法を上回る')
        print('  推奨論文軸: 元の主張を強化 (GLM/HierBayes を比較群に追加)')
        print('  投稿候補 : SIGSPATIAL Application Track (元の予定通り)')
    else:
        scenario = 'W'
        print('  ★ シナリオ W: Naive baseline (B2/B5) が最強')
        print('  推奨論文軸: 「なぜ rolling average が学習ベース手法を超えるか」')
        print('  (稀少事象では空間的近接性より時間的近接性が重要という発見)')
        print('  投稿候補 : SIGSPATIAL Negative Results / Insights Track')

    # Bootstrap 有意性 check (HierBayes vs B5, K=20)
    def _get_pval(boot_df, compA, compB, K=20):
        comp = f'{compA} vs {compB}'
        sub = boot_df[(boot_df['Comparison'] == comp) & (boot_df['K'] == K)]
        return float(sub['p_boot'].values[0]) if len(sub) > 0 else float('nan')

    p_hb_vs_b5_y = _get_pval(boot_df_y, 'HierBayes', 'B5: B2+B3')
    p_hb_vs_ttm_y = _get_pval(boot_df_y, 'HierBayes', 'TTM')
    p_gl_vs_ttm_y = _get_pval(boot_df_y, 'GLM-Logit', 'TTM')

    print()
    print('  統計的有意性 (Yamagata, K=20, p_boot):')
    print(f'    HierBayes vs B5:  p={p_hb_vs_b5_y:.4f}  '
          f'{"有意" if p_hb_vs_b5_y < 0.05 else "非有意"}')
    print(f'    HierBayes vs TTM: p={p_hb_vs_ttm_y:.4f}  '
          f'{"有意" if p_hb_vs_ttm_y < 0.05 else "非有意"}')
    print(f'    GLM-Logit vs TTM: p={p_gl_vs_ttm_y:.4f}  '
          f'{"有意" if p_gl_vs_ttm_y < 0.05 else "非有意"}')

    print()
    print(f'  Bonferroni 補正後有意水準: α = {ALPHA_BONF:.6f}')
    print('=' * 70)
    return scenario


verdict = auto_verdict(yama_res, akita_res, yama_boot_df, akita_boot_df)

# verdict.txt に保存
with open(OUTPUT_DIR + '/verdict.txt', 'w', encoding='utf-8') as _vf:
    _vf.write(f'Scenario: {verdict}\n')
    _vf.write('\nYamagata Recall@20:\n')
    for m in METHOD_ORDER:
        if m in yama_res.index:
            _vf.write(f'  {m}: {yama_res.loc[m,"Recall@20"]:.4f}\n')
print(f'\nverdict.txt 保存: {OUTPUT_DIR}/verdict.txt')

シナリオ自動判定
  [Yamagata Recall@20]
    GLM-Logit                : 0.5470
    HierBayes                : 0.5453
    B5: B2+B3                : 0.5342
    B1: Static prior         : 0.5331
    B4: B1+B3                : 0.5171
    TTM                      : 0.4917
    B2: Recent MA            : 0.4857
    B3: DoY season           : 0.4749
    ET                       : 0.4739
    B0: Random               : 0.1256
    Poisson-GLM              : 0.0267
    Poisson-GLM-cs           : 0.0267

  ★ シナリオ X: GLM が機械学習手法 (TTM/ET) を上回る
  推奨論文軸: 「Foundation model は古典統計を超えていない」
  投稿候補 : SIGSPATIAL Vision Track / NeurIPS Workshop

  統計的有意性 (Yamagata, K=20, p_boot):
    HierBayes vs B5:  p=0.3701  非有意
    HierBayes vs TTM: p=0.0004  有意
    GLM-Logit vs TTM: p=0.0002  有意

  Bonferroni 補正後有意水準: α = 0.000427

verdict.txt 保存: /content/drive/MyDrive/ttm_bear_glm_results/verdict.txt


In [19]:
# ── 論文用 Markdown テーブル一括出力 + Drive 保存 ────────────────────────────────

def df_to_md(df, title, key_cols=None):
    if key_cols:
        df = df[[c for c in key_cols if c in df.columns]]
    lines = [f'## {title}', '']
    header = '| Method | ' + ' | '.join(df.columns) + ' |'
    sep    = '|--------|' + '|'.join('---' for _ in df.columns) + '|'
    lines += [header, sep]
    for idx, row in df.iterrows():
        vals = ' | '.join(f'{v:.4f}' if isinstance(v, float) and v == v else str(v)
                           for v in row.values)
        lines.append(f'| {idx:22s} | {vals} |')
    return '\n'.join(lines)


def boot_to_md(df, title, K=20):
    sub = df[df['K'] == K]
    lines = [f'## {title} (K={K})', '']
    lines += ['| Comparison | obs | 95% CI | p_boot | p_perm | sig_boot | sig_bonf |',
               '|------------|-----|--------|--------|--------|---------|---------|']
    for _, r in sub.iterrows():
        lines.append(f'| {r["Comparison"]:35s} | {r["obs"]:+.4f} | '
                      f'[{r["ci_lo"]:+.4f},{r["ci_hi"]:+.4f}] | '
                      f'{r["p_boot"]:.4f} | {r["p_perm"]:.4f} | '
                      f'{r["sig_boot"]} | {r["sig_bonf"]} |')
    return '\n'.join(lines)


main_cols = ['Recall@10', 'Recall@20', 'Recall@30', 'ROC-AUC', 'PR-AUC',
              'Brier', 'BSS', 'ECE']

yama_md = df_to_md(yama_res, 'Yamagata — 全手法結果', main_cols)
akita_md = df_to_md(akita_res, 'Akita — 全手法結果', main_cols)
yama_boot_md = boot_to_md(yama_boot_df, 'Yamagata Bootstrap CI')
akita_boot_md = boot_to_md(akita_boot_df, 'Akita Bootstrap CI')

print('=' * 70)
print(yama_md)
print()
print(akita_md)
print()
print(yama_boot_md[:2000])  # truncate for display

# Drive に保存
for fname, content in [
    ('results_table_yamagata.md', yama_md),
    ('results_table_akita.md',    akita_md),
    ('bootstrap_ci_yamagata.md',  yama_boot_md),
    ('bootstrap_ci_akita.md',     akita_boot_md),
]:
    with open(OUTPUT_DIR + '/' + fname, 'w', encoding='utf-8') as _f:
        _f.write(content)

print(f'\nMarkdown テーブル保存完了: {OUTPUT_DIR}/')

# 層別結果も保存
strat_md_y = df_to_md(yama_strat.reset_index().set_index('Method'),
                       'Yamagata 層別 Recall@20 (Low/Medium/High)')
strat_md_a = df_to_md(akita_strat.reset_index().set_index('Method'),
                       'Akita 層別 Recall@20 (Low/Medium/High)')
with open(OUTPUT_DIR + '/stratified_recall_yamagata.md', 'w', encoding='utf-8') as _f:
    _f.write(strat_md_y)
with open(OUTPUT_DIR + '/stratified_recall_akita.md', 'w', encoding='utf-8') as _f:
    _f.write(strat_md_a)
print('層別結果も保存完了')

## Yamagata — 全手法結果

| Method | Recall@10 | Recall@20 | Recall@30 | ROC-AUC | PR-AUC | Brier | BSS | ECE |
|--------|---|---|---|---|---|---|---|---|
| B0: Random             | 0.0602 | 0.1256 | 0.1860 | 0.5022 | 0.0720 | 0.3343 | -8.0621 | 0.4625 |
| B1: Static prior       | 0.2864 | 0.5331 | 0.6588 | 0.5000 | 0.0614 | 0.0366 | 0.0070 | 0.0292 |
| B2: Recent MA          | 0.3106 | 0.4857 | 0.6066 | 0.6932 | 0.1368 | 0.0314 | 0.1478 | 0.0092 |
| B3: DoY season         | 0.3048 | 0.4749 | 0.5871 | 0.6631 | 0.1206 | 0.0355 | 0.0381 | 0.0292 |
| B4: B1+B3              | 0.3204 | 0.5171 | 0.6441 | 0.4994 | 0.0897 | 0.1445 | -2.9168 | 0.1561 |
| B5: B2+B3              | 0.3326 | 0.5342 | 0.6604 | 0.5497 | 0.1003 | 0.1253 | -2.3964 | 0.1330 |
| ET                     | 0.2927 | 0.4739 | 0.6066 | 0.7057 | 0.1196 | 0.0971 | -1.6337 | 0.1130 |
| TTM                    | 0.2906 | 0.4917 | 0.6201 | 0.6178 | 0.1021 | 0.0362 | 0.0194 | 0.0284 |
| GLM-Logit              | 0.3448 | 0.5470 | 0.6904 | 

In [20]:
import platform

_t_total = _tm.time() - _t0_global
print('=' * 60)
print('実行完了サマリー')
print('=' * 60)
print(f'総実行時間 : {_t_total / 60:.1f} 分 ({_t_total:.0f} 秒)')
print(f'Python     : {sys.version.split()[0]}')
print(f'numpy      : {np.__version__}')
print(f'pandas     : {pd.__version__}')
try:
    import pymc; print(f'PyMC       : {pymc.__version__}')
except ImportError:
    print('PyMC       : 未インストール (HierBayes フォールバック使用)')
try:
    import numpyro; print(f'numpyro    : {numpyro.__version__}')
except ImportError:
    print('numpyro    : 未インストール')
print(f'Platform   : {platform.platform()}')
print()
print('出力ファイル:')
import os
for root, dirs, files in os.walk(OUTPUT_DIR):
    for fn in files:
        fp = os.path.join(root, fn)
        sz = os.path.getsize(fp) // 1024
        print(f'  {fp}  ({sz} KB)')
print()
print('=' * 60)
print('DONE. 上記シナリオ判定に基づき論文の主張軸を再設計してください。')
print('=' * 60)

実行完了サマリー
総実行時間 : 110.9 分 (6655 秒)
Python     : 3.12.13
numpy      : 2.0.2
pandas     : 2.2.2
PyMC       : 5.28.4
numpyro    : 0.21.0
Platform   : Linux-6.6.122+-x86_64-with-glibc2.35

出力ファイル:
  /content/drive/MyDrive/ttm_bear_glm_results/partial_pooling_visualization.png  (94 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/forest_plot_combined.png  (109 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/verdict.txt  (0 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/results_table_yamagata.md  (1 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/results_table_akita.md  (1 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/bootstrap_ci_yamagata.md  (1 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/bootstrap_ci_akita.md  (1 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/stratified_recall_yamagata.md  (0 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/stratified_recall_akita.md  (0 KB)
  /content/drive/MyDrive/ttm_bear_glm_results/traces/yamagata_hier_trace.pkl  (11902 KB)
  /con

---
## Part 7: Table 3 再計算 — 確信度フィルタリング (Day-level)

**論文 Table 3** の再計算。  
`heterogeneity_final_analysis_colab.ipynb` では誤った B5 実装 (L1-norm + Fourier) が
使用されていたため、このノートブックの正しい B5 実装 (Z-score + ±7日DoY窓) で
Table 3 を再計算する。

### 評価方法
- **Day-level 確信度フィルタリング**: 各日の確信度 = 全セルの平均確信度
  - 点予測系 (GLM, TTM, B1, B2, B5): `confidence = |score - 0.5|`
  - 階層ベイズ: `confidence = 1 / posterior_std`
- 上位 top_pct% の確信度を持つ日のみで `global_recall_at_k(K=20)` を計算

### 修正箇所まとめ
| 修正対象 | 誤 (旧) | 正 (新) |
|---|---|---|
| Table 1: HierBayes YGT R@20 | 0.545 | 0.542 |
| Table 3: B5 行 (top25/50/75/all) | 0.495/0.493/0.497/0.481 | (再計算) |
| Section 5.1: 差の範囲 | 0.2–2.3 pp | 0.5–2.3 pp |
| Section 4.1: 訓練期間 | (1,536 days) | (~2,284 days) |
| Section 6.3: B2 top-50% | 0.493 (B5の値を誤引用) | (再計算) |


In [21]:
# ── Part 7: Day-level 確信度フィルタリング関数 ─────────────────────────────────

def conf_filtered_recall_day(scores, labels, confidence, K=20, top_pct=1.0):
    """
    Day-level 確信度フィルタリングで Recall@K を計算する (論文 Table 3 の定義)。

    Args:
        scores:     (T, n_cells) float  -- 予測スコア
        labels:     (T, n_cells) bool/int -- 正解ラベル
        confidence: (T, n_cells) float  -- 確信度 (高いほど確信)
                    点予測系 = |score - 0.5|, HierBayes = 1/posterior_std
        K:          Top-K 評価 (論文デフォルト K=20)
        top_pct:    使用する日の割合 (例: 0.25 = 確信度上位 25% の日のみ)

    手順:
        1. 各日の確信度 = confidence.mean(axis=1)  → (T,) に集約
        2. 上位 top_pct 割の日インデックスを取得
        3. 選んだ日のみで global_recall_at_k を計算
    """
    day_conf  = confidence.mean(axis=1)          # (T,) → 各日の平均確信度
    n_keep    = max(1, int(np.ceil(len(day_conf) * top_pct)))
    keep_days = np.argsort(-day_conf)[:n_keep]   # 確信度の高い順に n_keep 日
    return global_recall_at_k(scores[keep_days], labels[keep_days], K)


def make_table3_row(method_name, scores, labels, confidence, K=20,
                    pcts=(0.25, 0.50, 0.75, 1.0)):
    """Table 3 の 1 行 (top25/50/75/All) を計算して辞書で返す。"""
    row = {'Method': method_name}
    for pct in pcts:
        col = 'All days' if pct >= 1.0 else f'top {int(pct*100)}%'
        row[col] = conf_filtered_recall_day(scores, labels, confidence,
                                             K=K, top_pct=pct)
    return row


def point_conf(scores):
    """点予測系の確信度: |score - 0.5| (高いほど 0 or 1 に近い)。"""
    return np.abs(scores.astype(np.float64) - 0.5)


def hier_conf(std_sc):
    """HierBayes 確信度: 1 / posterior_std (std が低いほど確信度が高い)。"""
    return 1.0 / (std_sc.astype(np.float64) + 1e-6)


print('Day-level 確信度フィルタリング関数定義 OK')


Day-level 確信度フィルタリング関数定義 OK


In [22]:
# ── Table 3 計算 ──────────────────────────────────────────────────────────────
# 対象: 山形県 (YGT)、K=20、top 25/50/75/100% の 4 列

PCTS      = (0.25, 0.50, 0.75, 1.0)
COL_NAMES = ['top 25%', 'top 50%', 'top 75%', 'All days']
K_T3      = 20

rows_t3 = []

# 1. HierBayes (確信度 = 1/posterior_std)
if 'yama_hier_std' in dir() and yama_hier_std is not None \
        and 'yama_hier_mean' in dir() and yama_hier_mean is not None:
    rows_t3.append(make_table3_row(
        'HierBayes', yama_hier_mean, yama_test_L,
        hier_conf(yama_hier_std), K=K_T3, pcts=PCTS))
    print('HierBayes: 計算完了')
else:
    print('⚠ HierBayes trace なし — HierBayes 行をスキップ (USE_MCMC=True が必要)')

# 2. GLM-Logit (確信度 = |score - 0.5|)
rows_t3.append(make_table3_row(
    'GLM-Logit', yama_glm_logit, yama_test_L,
    point_conf(yama_glm_logit), K=K_T3, pcts=PCTS))
print('GLM-Logit: 計算完了')

# 3. TTM (確信度 = |score - 0.5|)
rows_t3.append(make_table3_row(
    'TTM', yama_ttm_scores, yama_test_L,
    point_conf(yama_ttm_scores), K=K_T3, pcts=PCTS))
print('TTM: 計算完了')

# 4. B5: B2+B3 — benchmark実装 (Z-score + ±7日DoY窓) ★修正済み
#    旧 heterogeneity実装: L1-norm + Fourier harmonic → 0.495/0.493/0.497/0.481
#    正 benchmark実装:   Z-score + ±7日DoY窓          → 本セルで再計算
rows_t3.append(make_table3_row(
    'B5: B2+B3', yama_baselines['B5: B2+B3'], yama_test_L,
    point_conf(yama_baselines['B5: B2+B3']), K=K_T3, pcts=PCTS))
print('B5 (benchmark): 計算完了')

# 5. B1: Static prior (確信度 = |score - 0.5|)
rows_t3.append(make_table3_row(
    'B1: Static prior', yama_baselines['B1: Static prior'], yama_test_L,
    point_conf(yama_baselines['B1: Static prior']), K=K_T3, pcts=PCTS))
print('B1: 計算完了')

# 6. B2: Recent MA (確信度 = |score - 0.5|) ★新規追加行
#    Section 6.3 で誤って 0.493 (B5の値) と引用されていたため正しい値を確認
rows_t3.append(make_table3_row(
    'B2: Recent MA', yama_baselines['B2: Recent MA'], yama_test_L,
    point_conf(yama_baselines['B2: Recent MA']), K=K_T3, pcts=PCTS))
print('B2: 計算完了')

# ── DataFrame 化 ─────────────────────────────────────────────────────────────
table3_df = pd.DataFrame(rows_t3).set_index('Method')
table3_df.columns = COL_NAMES

print('\n=== Table 3 再計算結果 (Recall@20, Yamagata, benchmark B5実装) ===')
print(table3_df.to_string(float_format='{:.3f}'.format))


HierBayes: 計算完了
GLM-Logit: 計算完了
TTM: 計算完了
B5 (benchmark): 計算完了
B1: 計算完了
B2: 計算完了

=== Table 3 再計算結果 (Recall@20, Yamagata, benchmark B5実装) ===
                  top 25%  top 50%  top 75%  All days
Method                                               
HierBayes           0.875    0.639    0.555     0.542
GLM-Logit           0.889    0.619    0.548     0.547
TTM                 0.630    0.523    0.495     0.492
B5: B2+B3           0.542    0.541    0.531     0.534
B1: Static prior    1.000    0.592    0.544     0.533
B2: Recent MA       0.150    0.416    0.458     0.486


In [23]:
# ── Table 3 Markdown 出力 + 論文修正情報 ──────────────────────────────────────

METHOD_ORDER_T3 = [
    'HierBayes', 'GLM-Logit', 'TTM',
    'B5: B2+B3', 'B1: Static prior', 'B2: Recent MA'
]

print('## 論文 Table 3 (修正版) — Confidence-Filtered Recall@20 (Yamagata)')
print()
md_lines = [
    '| Method          | top 25% | top 50% | top 75% | All days |',
    '|:----------------|--------:|--------:|--------:|---------:|',
]
for m in METHOD_ORDER_T3:
    if m in table3_df.index:
        r    = table3_df.loc[m]
        note = '  *corrected*' if m == 'B5: B2+B3' else \
               '  *new row*'   if m == 'B2: Recent MA' else ''
        md_lines.append(
            f'| {m:<15} | {r["top 25%"]:.3f}    | {r["top 50%"]:.3f}    '
            f'| {r["top 75%"]:.3f}    | {r["All days"]:.3f}      |{note}')
print('\n'.join(md_lines))

print()
print('=' * 65)
print('論文修正サマリー')
print('=' * 65)

# --- Table 1: HierBayes R@20 ---
if 'HierBayes' in table3_df.index:
    h_all = table3_df.loc['HierBayes', 'All days']
    diff  = abs(h_all - 0.545)
    print(f'\n[Table 1] HierBayes YGT Recall@20')
    print(f'  論文値: 0.545  →  計算値: {h_all:.3f}')
    if diff > 0.001:
        print(f'  → 修正必要: 0.545 → {h_all:.3f}')
    else:
        print('  → 誤差 ≤ 0.001 のため修正不要')

# --- Table 3: B5 行 ---
if 'B5: B2+B3' in table3_df.index:
    b5 = table3_df.loc['B5: B2+B3']
    print(f'\n[Table 3] B5: B2+B3 行')
    print(f'  旧 (heterogeneity実装): 0.495 / 0.493 / 0.497 / 0.481')
    print(f'  新 (benchmark実装):     '
          f'{b5["top 25%"]:.3f} / {b5["top 50%"]:.3f} / '
          f'{b5["top 75%"]:.3f} / {b5["All days"]:.3f}')

# --- Section 6.3: B2 vs B5 誤記 ---
if 'B2: Recent MA' in table3_df.index and 'GLM-Logit' in table3_df.index:
    b2_top50  = table3_df.loc['B2: Recent MA', 'top 50%']
    glm_top50 = table3_df.loc['GLM-Logit', 'top 50%']
    gap       = glm_top50 - b2_top50
    print(f'\n[Section 6.3] B2 top-50% 値と gap')
    print(f'  旧テキスト: "exceeding B2\'s ...Recall@20 of 0.493 ...gap (0.146)"')
    print(f'    ※ 0.493 は B5 の top-50% 値を誤引用')
    print(f'  正しい B2 top-50%: {b2_top50:.3f}')
    print(f'  GLM-Logit top-50%: {glm_top50:.3f}')
    print(f'  正しい gap:        {gap:.3f}')
    print(f'  新テキスト: "exceeding B2\'s ...Recall@20 of {b2_top50:.3f} ...gap ({gap:.3f})"')

# --- その他の修正 ---
print(f'\n[Section 5.1] 精度差の範囲')
print(f'  旧: "0.2-2.3 percentage points"')
print(f'  新: "0.5-2.3 percentage points"  (最小差は GLM vs HierBayes の 0.5 pp)')

print(f'\n[Section 4.1] 訓練期間')
print(f'  旧: "(1,536 days)"  ← TTM の context window')
print(f'  新: "(~2,284 days)"  ← 実際の訓練期間 2018-10-01 〜 2024-12-31')

print('\n全修正確認完了')


## 論文 Table 3 (修正版) — Confidence-Filtered Recall@20 (Yamagata)

| Method          | top 25% | top 50% | top 75% | All days |
|:----------------|--------:|--------:|--------:|---------:|
| HierBayes       | 0.875    | 0.639    | 0.555    | 0.542      |
| GLM-Logit       | 0.889    | 0.619    | 0.548    | 0.547      |
| TTM             | 0.630    | 0.523    | 0.495    | 0.492      |
| B5: B2+B3       | 0.542    | 0.541    | 0.531    | 0.534      |  *corrected*
| B1: Static prior | 1.000    | 0.592    | 0.544    | 0.533      |
| B2: Recent MA   | 0.150    | 0.416    | 0.458    | 0.486      |  *new row*

論文修正サマリー

[Table 1] HierBayes YGT Recall@20
  論文値: 0.545  →  計算値: 0.542
  → 修正必要: 0.545 → 0.542

[Table 3] B5: B2+B3 行
  旧 (heterogeneity実装): 0.495 / 0.493 / 0.497 / 0.481
  新 (benchmark実装):     0.542 / 0.541 / 0.531 / 0.534

[Section 6.3] B2 top-50% 値と gap
  旧テキスト: "exceeding B2's ...Recall@20 of 0.493 ...gap (0.146)"
    ※ 0.493 は B5 の top-50% 値を誤引用
  正しい B2 top-50%: 0.416
  GLM-Logit to